# H3: what can be repaired after pre-training, and at what price

The claim is not that nothing can be done. It is that every repair available at
inference time has a cost that can be named, and that the one conceptually cleanest
repair makes things worse, because the vulnerability and the safety mechanism are the
same thing.

Recomputed from `results/h3_repair/`, plus the benchmark survey that explains why none
of this was visible before.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np, pandas as pd
from common import paths, metrics, tables, stats, quiet
# Narrow filter, not filterwarnings("ignore"): numeric warnings stay visible,
# because one of them hid a divide-by-zero in the tail-index inversion for months.
quiet.silence()
pd.set_option("display.width", 200)
R = lambda n: pd.read_csv(paths.result(n))
print("repo root:", paths.ROOT)


repo root: D:\faks\master\tabular-foundation-model-tails


## 1. Transforming the target before it reaches the model

Benefit is the response to one injected row, so smaller is better. Cost is the pinball
loss on clean data at each level.

In [2]:
rp = tables.ok_rows(R("repair.csv"))
print("%d runs, variants %s, models %s"
      % (len(rp), sorted(rp.variant.unique()), sorted(rp.model.unique())))
print()
print("benefit: response of Q(0.99) and Q(0.999) to one injected row")
print(rp.pivot_table(index="variant", columns="model",
                     values="response_q99", aggfunc="median").round(3).to_string())
print()
print("cost on clean data, pinball by level (relative to the raw variant):")
base = rp[rp.variant == "raw"].groupby("model")[["pb50", "pb90", "pb99", "pb999"]].median()
for v in sorted(rp.variant.unique()):
    if v == "raw":
        continue
    cur = rp[rp.variant == v].groupby("model")[["pb50", "pb90", "pb99", "pb999"]].median()
    rel = ((cur - base) / base * 100).round(1)
    print()
    print("  variant:", v)
    print(rel.to_string())

100 runs, variants ['asinh', 'log', 'rank', 'raw', 'winsor'], models ['TabICLv2', 'TabPFN-V3']

benefit: response of Q(0.99) and Q(0.999) to one injected row
model    TabICLv2  TabPFN-V3
variant                     
asinh       0.278      0.060
log         0.131      0.057
rank        0.031      0.031
raw        -0.382      0.629
winsor      0.045      0.023

cost on clean data, pinball by level (relative to the raw variant):

  variant: asinh
           pb50  pb90  pb99  pb999
model                             
TabICLv2   -0.2  -1.2  -3.8   19.8
TabPFN-V3  -0.2  -1.1  -3.5  119.3

  variant: log
           pb50  pb90  pb99  pb999
model                             
TabICLv2   -0.2  -1.6  -3.7   48.1
TabPFN-V3  -0.4  -1.6  -4.2   71.3

  variant: rank
           pb50  pb90  pb99  pb999
model                             
TabICLv2   -0.2  -1.4  -1.9   93.7
TabPFN-V3  -0.4  -1.5  -2.7   73.5

  variant: winsor
           pb50  pb90  pb99  pb999
model                             
TabICLv2  

## 2. Replacing the scale estimator instead of the data

The conceptually cleanest repair: swap the standard deviation for a robust scale. It
makes every level worse by an order of magnitude, because the robust scale is a small
fraction of the standard deviation, so values land outside the bin grid the model was
pre-trained on. This is the negative result the thesis keeps.

In [3]:
rs = tables.ok_rows(R("robust_scale.csv"))
print("%d runs, variants %s" % (len(rs), sorted(rs.variant.unique())))
print()
cols = [c for c in ["response_q99", "response_q999", "pb50", "pb99", "pb999",
                    "scale_ratio"] if c in rs.columns]
print(rs.pivot_table(index="variant", values=cols, aggfunc="median").round(3).to_string())
print()
if "scale_ratio" in rs.columns:
    r = rs[rs.variant != "raw"].scale_ratio.median()
    print("robust scale is about %.3f of the standard deviation" % r)
    print("so the contaminated values fall outside the pre-trained support")

40 runs, variants ['raw', 'robust']

          pb50   pb99  pb999  response_q99  response_q999  scale_ratio
variant                                                               
raw      2.149  1.538  0.321        -0.340          0.951          NaN
robust   2.940  2.378  1.775         9.247          6.092        0.053

robust scale is about 0.053 of the standard deviation
so the contaminated values fall outside the pre-trained support


## 3. On real datasets the recommendation is mixed

Winsorizing helps at 0.99 under contamination but consistently hurts one model at 0.999.
This is the honest summary, and it is weaker than the synthetic result alone suggests.

In [4]:
rr = tables.ok_rows(R("repair_real.csv"))
print("%d runs, %d datasets, doses %s"
      % (len(rr), rr.dataset.nunique(), sorted(rr.dose.unique())))
base = rr[rr.variant == "raw"].groupby(["dataset", "model", "dose"])[["pb99", "pb999"]].median()
win = rr[rr.variant != "raw"].groupby(["dataset", "model", "dose"])[["pb99", "pb999"]].median()
rel = ((win - base) / base * 100).dropna()
for lvl, lab in [("pb99", "alpha = 0.99"), ("pb999", "alpha = 0.999")]:
    print()
    print("change in pinball from winsorizing, %s (negative is better):" % lab)
    print(rel[lvl].unstack("model").round(1).to_string())

270 runs, 3 datasets, doses [np.float64(0.0), np.float64(10.0), np.float64(100.0)]

change in pinball from winsorizing, alpha = 0.99 (negative is better):
model                            GBM  TabICLv2  TabPFN-V3
dataset                  dose                            
Allstate_Claims_Severity 0.0     3.1      12.1       10.9
                         10.0   -9.0      19.3        4.1
                         100.0  -9.0      10.7       -4.2
OnlineNewsPopularity     0.0     3.4      -2.0        1.2
                         10.0  -11.1      -3.2       -8.2
                         100.0 -22.1     -13.1      -19.5
freMTPL2sev              0.0    -1.5     -11.7        0.0
                         10.0  -10.1      -6.2       -2.3
                         100.0 -23.5       2.2       -0.4

change in pinball from winsorizing, alpha = 0.999 (negative is better):
model                            GBM  TabICLv2  TabPFN-V3
dataset                  dose                            
Allstate_Claims_Se

## 4. Correcting the mean afterwards

A scaling factor fitted on validation and applied to the test predictions. The share of
the deficit it recovers is only interpretable where a deficit exists in the first place:
where the raw ratio is already about 1, the denominator is near zero and the share is
meaningless.

In [5]:
mc = R("mean_correction.csv")
syn = mc[mc.dataset.str.startswith("sint")]
t = syn.groupby("model")[["ratio_raw", "ratio_constant", "ratio_mean_ratio",
                          "ratio_isotonic"]].median()
print("synthetic data, median model/true mean ratio:")
print(t.round(3).to_string())
print()
print("share of the deficit recovered, only where a deficit exists (raw < 0.95):")
for m, row in t.iterrows():
    if row.ratio_raw >= 0.95:
        print("  %-10s raw ratio %.3f, no deficit to recover" % (m, row.ratio_raw))
        continue
    for c in ["ratio_constant", "ratio_mean_ratio", "ratio_isotonic"]:
        share = (row[c] - row.ratio_raw) / (1 - row.ratio_raw)
        print("  %-10s %-18s %.0f%%" % (m, c, 100 * share))
print()
print("The old README quoted 72 to 74% recovered. No finding documents that number and")
print("it is not reproduced here, so it has been removed rather than restated.")

synthetic data, median model/true mean ratio:
           ratio_raw  ratio_constant  ratio_mean_ratio  ratio_isotonic
model                                                                 
GBM            1.029           1.571             0.979           0.936
TabDPT         1.006           1.045             1.048           1.038
TabICLv2       1.027           1.039             0.970           0.964
TabPFN-V2      0.773           0.958             0.964           0.978
TabPFN-V3      0.824           0.991             0.965           0.981

share of the deficit recovered, only where a deficit exists (raw < 0.95):
  GBM        raw ratio 1.029, no deficit to recover
  TabDPT     raw ratio 1.006, no deficit to recover
  TabICLv2   raw ratio 1.027, no deficit to recover
  TabPFN-V2  ratio_constant     81%
  TabPFN-V2  ratio_mean_ratio   84%
  TabPFN-V2  ratio_isotonic     90%
  TabPFN-V3  ratio_constant     95%
  TabPFN-V3  ratio_mean_ratio   80%
  TabPFN-V3  ratio_isotonic     89%

The old R

## 5. Grafting an extreme-value tail

Pre-registered as a repair and abandoned by its own criterion: the benefit does not
survive an honest gate.

In [6]:
try:
    eg = tables.ok_rows(R("evt_graft.csv"))
    print("%d runs, variants %s" % (len(eg), sorted(eg.variant.unique())))
    cols = [c for c in ["twcrps", "pinball99", "pinball999", "crps"] if c in eg.columns]
    print(eg.pivot_table(index="variant", values=cols, aggfunc="median").round(4).to_string())
except Exception as e:
    print("evt_graft.csv:", e)

180 runs, variants ['EVT regularised', 'EVT simple', 'raw']
                    crps  pinball99  pinball999  twcrps
variant                                                
EVT regularised  19.0631     3.8087      1.3790  7.8319
EVT simple       19.0629     3.8121      1.3790  7.8318
raw              19.1331     4.0069      1.3638  7.8959


## 6. Why none of this was visible before

The public benchmark stops just short of where these effects live: its scores stop at
alpha = 0.99, each fold holds 600 test points, and only a fifth of its datasets have a
tail index that three estimators agree on.

In [7]:
sb = R("scoringbench_survey.csv")
print("%d datasets surveyed" % len(sb))
if "passes" in sb.columns:
    n_pass = int(sb.passes.fillna(False).astype(bool).sum())
    print("with an identified tail index: %d of %d (%.0f%%)"
          % (n_pass, len(sb), 100 * n_pass / len(sb)))
est = [c for c in ["gpd10", "hill", "quantile"] if c in sb.columns]
print("tail estimates available:", {c: int(sb[c].notna().sum()) for c in est})
print()
ds = R("dataset_selection.csv")
if "passes" in ds.columns:
    print("hand-picked heavy-tailed candidates: %d of %d pass the agreement test"
          % (int(ds.passes.fillna(False).astype(bool).sum()), len(ds)))
    print(ds[["dataset", "passes"]].to_string(index=False))

102 datasets surveyed
with an identified tail index: 19 of 102 (19%)
tail estimates available: {'gpd10': 96, 'hill': 96, 'quantile': 96}



hand-picked heavy-tailed candidates: 5 of 13 pass the agreement test
                     dataset  passes
                 freMTPL2sev    True
   Buzzinsocialmedia_Twitter    True
                    diamonds    True
     delays_zurich_transport    True
    Allstate_Claims_Severity    True
                black_friday   False
     nyc-taxi-green-dec-2016   False
             medical_charges   False
        Airlines_DepDelay_1M   False
SGEMM_GPU_kernel_performance   False
         Bike_Sharing_Demand   False
           video_transcoding   False
                wine_quality   False


## 7. The context arms on the generator

`clip_context.csv`. Five runs of arms, all paired seed by seed against `raw` on the
same cell, which is the only way these are comparable: a median over arms measured
on different cell sets is what made a 3% gap in the threshold sweep turn out to be
0.06% once it was paired.

`sd_cap` derives the cap from the context's own sd shift instead of picking a
multiple of the robust sd; the four `rank_*` arms replace the target by its own
scores so the leverage cannot enter at all.

In [ ]:
cc = tables.ok_rows(R("h3_repair/clip_context.csv"))
K = ["xi", "sd_shift_target", "seed"]
print("%d runs, variants %s" % (len(cc), sorted(cc.variant.unique())))
print("cells per variant:", cc.variant.value_counts().to_dict())

def _arm(d, arm, c=200.0):
    """One arm as a frame. `clip` carries three caps per cell, so it needs the one
    the comparison is about; every other arm has a single row per cell."""
    s = d[d.variant == arm]
    return s[s.clip_c == c] if arm == "clip" else s

def paired(d, arm, ref, col, key=K):
    """Wins and median relative change against a reference arm, on shared cells."""
    a = _arm(d, arm).set_index(key)[col]
    b = _arm(d, ref).set_index(key)[col]
    j = pd.concat([b.rename("ref"), a.rename("arm")], axis=1, join="inner").dropna()
    rel = (j["arm"] - j["ref"]) / j["ref"]
    return len(j), int((j["arm"] < j["ref"]).sum()), float(rel.median())

print()
print("pinball at 0.999 against raw, by arm and dose (wins / n, median change)")
for shift in sorted(cc.sd_shift_target.unique()):
    s = cc[cc.sd_shift_target == shift]
    row = []
    for arm in ["clip", "sd_cap", "rank_gpd", "rank_gpd_trim", "rank_exp"]:
        n, w, rel = paired(s, arm, "raw", "pb999")
        row.append(f"{arm} {w}/{n} {rel:+.1%}")
    print(f"  shift {shift:>5}: " + "   ".join(row))

print()
print("R3: the rank arms keep the border grid inside the data, by construction")
print(cc.pivot_table(index="variant", columns="sd_shift_target",
                     values="borders_in_data", aggfunc="median").round(0).to_string())
print()
print("implied xi (true xi in the index); a positive value is a tail that survived")
print(cc.pivot_table(index=["xi", "variant"], columns="sd_shift_target",
                     values="xi_implied", aggfunc="median").round(3).to_string())

### The clean-context price

P9 claimed `sd_cap` costs exactly nothing where there is nothing to repair. As a
construction claim that holds; as an empirical one it needs the caveat that a clean
heavy-tailed context is often *not* below the threshold, so the arm still clips.

In [ ]:
clean = cc[cc.sd_shift_target == 1.0]
print("on a clean context: context values changed, and the price at each level")
for arm in ["clip", "sd_cap", "rank_emp", "rank_gpd", "rank_gpd_trim", "rank_exp"]:
    touched = _arm(clean, arm).n_clipped.median()
    bits = []
    for lv in ("99", "999"):
        n, w, rel = paired(clean, arm, "raw", f"pb{lv}")
        bits.append(f"{lv}: {w}/{n} {rel:+.1%}")
    print(f"  {arm:14s} n_clipped median {touched:8.0f}   " + "   ".join(bits))
print()
print("note: the rank arms rewrite every row, so n_clipped is n, not 0 --")
print("they are not comparable to a cap's count, only to each other.")

## 8. The same arms on real tables, and under a unit error

`clip_context_real.csv` places the arms in the leverage bins of the prevalence
survey; `unit_error_real.csv` multiplies one visible row by 100, so the corrupted
value *is* the top exceedance the tail fit would use.

This is where the fourth run's rank arm failed, and the number below is the one the
script's own docstring cites.

In [ ]:
ue = tables.ok_rows(R("h3_repair/unit_error_real.csv"))
KU = ["dataset", "model", "repeat", "factor"]
print("%d rows, arms %s" % (len(ue), sorted(ue.arm.unique())))
def cells_per_arm(d):
    """Rows per arm, counting `clip` once rather than once per cap: it is the only
    arm that carries several clip_c values on the same cell."""
    one = d[(d.arm != "clip") | (d.clip_c == 200.0)]
    return one.arm.value_counts()

counts = cells_per_arm(ue)
print("cells per arm:", counts.to_dict())
if counts.nunique() > 1:
    short = sorted(counts[counts < counts.max()].index)
    print(f"  !! INCOMPLETE: {short}; the pairing below uses shared cells only")

raw = ue[ue.arm == "raw"].set_index(KU)["pb999"]
print()
print("pb999 against raw: share of cells more than 100% worse, and the worst cell")
print("(the docstring cites 13.8% and 86000x for rank_gpd)")
for arm in ["clip", "sd_cap", "rank_emp", "rank_gpd", "rank_gpd_trim", "rank_exp"]:
    a = ue[ue.arm == arm]
    if arm == "clip":
        a = a[a.clip_c == 200.0]
    a = a.set_index(KU)["pb999"]
    j = pd.concat([raw.rename("ref"), a.rename("arm")], axis=1, join="inner").dropna()
    rel = (j["arm"] - j["ref"]) / j["ref"]
    print(f"  {arm:14s} n={len(j):4d}  >+100%: {float((rel > 1).mean()):6.2%}"
          f"   worst: {rel.max() * 100:,.0f}%   median: {rel.median():+.1%}")

In [ ]:
cr = tables.ok_rows(R("h3_repair/clip_context_real.csv"))
counts = cells_per_arm(cr)
print("clip_context_real.csv, cells per arm:", counts.to_dict())
if counts.nunique() > 1:
    short = sorted(counts[counts < counts.max()].index)
    print(f"  !! INCOMPLETE: {short} have fewer cells than the rest.")
    print("     Nothing below should be quoted for those arms until the run finishes.")
print()
print("d_q99 against the without-max reference, median by arm and model")
print(cr.pivot_table(index="arm", columns="model", values="d_q99",
                     aggfunc="median").round(3).to_string())

## 9. The output patch: conformal, with a fitted tail

`tail_splice.csv`. The bound is the model's quantile plus a quantile of the
calibration scores; `cqr_emp` reads that empirically, the `cqr_gpd_*` arms fit a
GPD above a threshold so the 0.999 level is reachable at all from 500 scores.

**Degenerate rows first.** An arm is degenerate on a cell when it fell back to
something that is not the method: `cqr_emp` past the reach of 500 scores, a
`cqr_gpd_*` whose GPD declined to fit, `cond_gpd` below its own threshold. Those
rows are the bare model wearing the arm's name, and pooling them into the arm's
median compares a method against a mixture of itself and no method at all.

In [ ]:
ts = tables.ok_rows(R("h3_repair/tail_splice.csv"))
ts = ts.assign(degenerate=ts.degenerate.fillna(False).astype(bool))
print("%d rows, arms %s" % (len(ts), sorted(ts.arm.unique())))
print("treatments:", sorted(ts.treatment.unique()))
print()
share = ts.pivot_table(index="arm", columns="level", values="degenerate", aggfunc="mean")
print("degenerate share by arm and level:")
print(share.round(3).to_string())
print()
print("Rows written before the fallback was flagged record a GPD arm that never")
print("fitted a GPD as degenerate=False; xi_hat is NaN on exactly those rows:")
nofit = ts[ts.arm.str.startswith("cqr_gpd")].groupby(["arm", "level"]).apply(
    lambda x: float(x.xi_hat.isna().mean()), include_groups=False).unstack()
print(nofit.round(3).to_string())
ts = ts[~ts.degenerate]

In [ ]:
KT = ["dataset", "model", "repeat", "factor", "treatment"]
print("T5/T6: does the GPD threshold matter? paired by dataset-model unit at 0.99")
at99 = ts[ts.level == 0.99]
ref = at99[at99.arm == "cqr_gpd_0.9"].set_index(KT)["pinball"]
for arm in sorted(a for a in at99.arm.unique() if a.startswith("cqr_gpd")):
    if arm == "cqr_gpd_0.9":
        continue
    a = at99[at99.arm == arm].set_index(KT)["pinball"]
    j = pd.concat([ref.rename("ref"), a.rename("arm")], axis=1, join="inner").dropna()
    j = j.groupby(level=[0, 1]).median()          # the free unit is a dataset-model pair
    rel = (j["arm"] - j["ref"]) / j["ref"]
    print(f"  {arm:14s} better in {int((j['arm'] < j['ref']).sum()):2d}/{len(j)} units,"
          f" median {rel.median():+.2%}")
print()
print("coverage at each level, by arm; nominal is the level itself")
print(ts.pivot_table(index="level", columns="arm", values="coverage",
                     aggfunc="median").round(4).to_string())
print()
print("T9: does the rank context make the patch redundant? pinball at 0.99 by treatment")
print(at99.pivot_table(index="treatment", columns="arm", values="pinball",
                       aggfunc="median").round(1).to_string())

## Disagreements with the text

In [8]:
notes = [
 "the 72 to 74% mean-deficit recovery in the old README is not reproduced by "
 "mean_correction.csv and is documented in no finding; it has been dropped.",
 "the repair is measured on two models synthetically and three on real data; TabDPT, "
 "TabFM and EXAONE are absent, and the CSV that the repair finding cites for the "
 "mean-only models does not exist. run/ carries drivers for both gaps.",
 "the fifth-run arms rank_gpd_trim and rank_exp are complete in clip_context.csv and "
 "unit_error_real.csv but NOT in clip_context_real.csv, where they have fewer cells "
 "than the other arms; section 8 prints the counts and refuses to quote them.",
 "an earlier draft of clip_context.py reported the worst rank_gpd cell at 0.999 as "
 "+51889%. Section 8 recomputes it as +8619507% (the 86000x the same docstring gives "
 "two paragraphs later); the smaller figure is not reproduced by any slice of the file.",
 "the umbrella claim that not everything can be repaired is an induction from two "
 "failed repairs and one priced one; the checkable content is the price per level, "
 "which is what the tables above report.",
]
for n in notes:
    print("-", n)

- the 72 to 74% mean-deficit recovery in the old README is not reproduced by mean_correction.csv and is documented in no finding; it has been dropped.
- the repair is measured on two models synthetically and three on real data; TabDPT, TabFM and EXAONE are absent, and the CSV that the repair finding cites for the mean-only models does not exist. run/ carries drivers for both gaps.
- the umbrella claim that not everything can be repaired is an induction from two failed repairs and one priced one; the checkable content is the price per level, which is what the tables above report.
